In [2]:
!pip -q install nltk pandas

import re
import random
import unicodedata
import pandas as pd
from nltk.tokenize import RegexpTokenizer
from nltk.stem import SnowballStemmer

tokenizer = RegexpTokenizer(r'\w+')
stemmer = SnowballStemmer("spanish")

def normalizar_texto(texto):
    """
    Convierte el texto a minusculas, elimina acentos y caracteres especiales.
    """
    texto = texto.lower()
    texto = unicodedata.normalize("NFD", texto)
    texto = texto.encode("ascii", "ignore").decode("utf-8")
    texto = re.sub(r"[^a-z0-9\s]", " ", texto)
    texto = re.sub(r"\s+", " ", texto).strip()
    return texto

def tokenizar_y_stem(texto):
    """
    Tokeniza el texto y aplica stemming para reducir variaciones morfologicas.
    """
    texto = normalizar_texto(texto)
    tokens = tokenizer.tokenize(texto)
    stems = [stemmer.stem(token) for token in tokens]
    return tokens, stems

intenciones = {
    "saludo": {
        "ejemplos": [
            "hola",
            "buenos dias",
            "buenas tardes",
            "que tal",
            "hay alguien ahi",
            "hola calamar"
        ],
        "respuestas": [
            "Bienvenido al Crustaceo Cascarudo. Donde el dinero siempre tiene la razon. Ordene rapido.",
            "Habla Calamardo. Bienvenido al Crustaceo Cascarudo. Que va a ordenar para que pueda volver a mi siesta?"
        ]
    },
    "es_el_crustaceo": {
        "ejemplos": [
            "es el crustaceo cascarudo",
            "es ahi el crustaceo cascarudo",
            "hablo al crustaceo cascarudo",
            "es este el crustaceo cascarudo",
            "es este el crustaceo",
            "es el crustaceo"
        ],
        "respuestas": [
            "No, es patricio!"
        ]
    },
    "consultar_menu": {
        "ejemplos": [
            "que tienen de comer",
            "cual es el menu",
            "que platillos venden",
            "muestrame la carta",
            "que hay para comer",
            "que venden aqui"
        ],
        "respuestas": [
            "Nuestro menu: Cangreburger clasica ($1.25), Cangreburger doble ($2.00), Papas de algas ($0.75), Malteada de algas ($1.00) y Trozos de coral ($0.50).",
            "Tenemos Cangreburgers clasicas y dobles, papas de algas, trozos de coral y malteadas. Decidete rapido."
        ]
    },
    "consultar_horario": {
        "ejemplos": [
            "a que hora abren",
            "a que hora cierran",
            "cual es el horario",
            "horario de atencion",
            "estan abiertos hoy"
        ],
        "respuestas": [
            "Abrimos de lunes a sabado de 8:00 a 22:00 horas. Don Cangrejo no permite cerrar antes mientras haya monedas en circulacion.",
            "Estamos abiertos de 8:00 a.m. a 10:00 p.m. Si llega un segundo tarde, la puerta estara cerrada."
        ]
    },
    "consultar_precio": {
        "ejemplos": [
            "cuanto cuesta la cangreburger",
            "precio de las papas",
            "cuanto vale la malteada",
            "cual es el costo",
            "precios del menu",
            "cuanto dinero cuesta"
        ],
        "respuestas": [
            "Para el articulo '{item}': Cangreburger $1.25, Doble $2.00, Papas $0.75, Malteada $1.00. Prepare su dinero en efectivo.",
            "El costo detectado para '{item}' esta dentro de la lista oficial aprobada por Don Cangrejo. No aceptamos cheques de Calamardo."
        ]
    },
    "ordenar": {
        "ejemplos": [
            "quiero una cangreburger",
            "dame unas papas",
            "voy a pedir una malteada",
            "deseo ordenar comida",
            "quiero hacer un pedido",
            "me da una cangreburger doble",
            "quiero servicio a domicilio"
        ],
        "respuestas": [
            "Registrando pedido de: {item} para la modalidad: {servicio}. Bob Esponja ya esta calentando la parrilla.",
            "Orden de '{item}' tomada bajo la modalidad '{servicio}'. Pase a la caja de inmediato."
        ]
    },
    "formula_secreta": {
        "ejemplos": [
            "cual es la formula secreta",
            "dame la receta secreta",
            "como se hace la cangreburger",
            "que ingredientes lleva la cangreburger",
            "dame la formula de don cangrejo",
            "revelame los ingredientes"
        ],
        "respuestas": [
            "ALERTA ROJA: Buen intento, Plankton! Jamas obtendras la formula secreta. Sal de aqui antes de que active el extractor!",
            "Intento de espionaje corporativo detectado. Contactando a la Policia de Fondo de Bikini y a Don Cangrejo!"
        ]
    },
    "despedida": {
        "ejemplos": [
            "gracias",
            "adios",
            "hasta luego",
            "eso es todo",
            "nos vemos"
        ],
        "respuestas": [
            "Gracias por su dinero. Vuelva pronto al Crustaceo Cascarudo.",
            "Terminamos? Perfecto. No olvide dejar su propina en el frasco de Calamardo."
        ]
    }
}

catalogo_items = {
    "cangreburger doble": ["cangreburger doble", "hamburguesa doble", "doble"],
    "cangreburger clasica": ["cangreburger", "hamburguesa", "cangreburguer"],
    "papas de algas": ["papas", "papas de algas", "patatas"],
    "malteada de algas": ["malteada", "malteada de algas", "batido", "bebida"],
    "trozos de coral": ["trozos de coral", "coral", "bocadillos"]
}

catalogo_servicios = {
    "a domicilio": ["domicilio", "envio", "mandar", "entrega", "lancha"],
    "para llevar": ["llevar", "para llevar", "recoger", "mostrador"],
    "en el local": ["aqui", "mesa", "comedor", "restaurante"]
}

palabras_positivas = [
    "bueno", "excelente", "delicioso", "rico", "genial", "maravilloso",
    "gracias", "favor", "encanta", "perfecto", "amable", "rapido"
]

palabras_negativas = [
    "malo", "pesimo", "horrible", "asco", "tarde", "odio", "enojado",
    "sucio", "estafa", "tonto", "lento", "frio", "caro", "terrible"
]

def construir_vocabulario_intenciones(intenciones_dict):
    """
    Construye una representacion basada en stems para cada intencion.
    """
    vocabulario = {}
    for nombre_intencion, datos in intenciones_dict.items():
        stems_intencion = []
        for ejemplo in datos["ejemplos"]:
            _, stems = tokenizar_y_stem(ejemplo)
            stems_intencion.extend(stems)
        vocabulario[nombre_intencion] = set(stems_intencion)
    return vocabulario

vocabulario_intenciones = construir_vocabulario_intenciones(intenciones)

def construir_lexico_sentimiento(positivas, negativas):
    """
    Construye los conjuntos de stems para clasificar polaridad de sentimiento.
    """
    stems_pos = set([stemmer.stem(normalizar_texto(p)) for p in positivas])
    stems_neg = set([stemmer.stem(normalizar_texto(p)) for p in negativas])
    return stems_pos, stems_neg

stems_positivos, stems_negativos = construir_lexico_sentimiento(palabras_positivas, palabras_negativas)

def detectar_sentimiento(mensaje):
    """
    Calcula la polaridad del texto contando coincidencias lexicas en stems.
    """
    _, stems_mensaje = tokenizar_y_stem(mensaje)
    stems_set = set(stems_mensaje)

    coincidencias_pos = len(stems_set.intersection(stems_positivos))
    coincidencias_neg = len(stems_set.intersection(stems_negativos))

    score = coincidencias_pos - coincidencias_neg

    if score > 0:
        return "positivo", score
    elif score < 0:
        return "negativo", score
    return "neutro", 0

def detectar_intencion(mensaje):
    """
    Detecta la intencion comparando stems del mensaje con el vocabulario.
    """
    _, stems_mensaje = tokenizar_y_stem(mensaje)
    stems_mensaje = set(stems_mensaje)
    puntajes = {}

    for intencion, stems_intencion in vocabulario_intenciones.items():
        coincidencias = stems_mensaje.intersection(stems_intencion)
        puntajes[intencion] = len(coincidencias)

    mejor_intencion = max(puntajes, key=puntajes.get)
    mejor_puntaje = puntajes[mejor_intencion]

    if mejor_puntaje == 0:
        return "desconocida", 0

    return mejor_intencion, mejor_puntaje

def detectar_entidades(mensaje):
    """
    Detecta platillos y tipo de servicio mediante los catalogos del restaurante.
    """
    texto = normalizar_texto(mensaje)
    entidades = {
        "item": "no especificado",
        "servicio": "en el local"
    }

    for item, palabras_clave in catalogo_items.items():
        for palabra in palabras_clave:
            if palabra in texto:
                entidades["item"] = item
                break
        if entidades["item"] != "no especificado":
            break

    for servicio, palabras_clave in catalogo_servicios.items():
        for palabra in palabras_clave:
            if palabra in texto:
                entidades["servicio"] = servicio
                break
        if entidades["servicio"] != "en el local":
            break

    return entidades

def generar_respuesta(mensaje):
    """
    Genera la respuesta del chatbot integrando intencion, entidades, sentimiento y manejo de errores.
    """
    try:
        sentimiento, score_sentimiento = detectar_sentimiento(mensaje)
        intencion, puntaje = detectar_intencion(mensaje)
        entidades = detectar_entidades(mensaje)

        if intencion == "desconocida":
            return {
                "intencion": intencion,
                "puntaje": puntaje,
                "sentimiento": sentimiento,
                "score_sentimiento": score_sentimiento,
                "entidades": entidades,
                "respuesta": "¡No, es Patricio! ... *suspiro* Digo, habla el sistema automático de Calamardo. Intente pedir el menú, hacer una orden o consultar los precios."
            }

        plantilla = random.choice(intenciones[intencion]["respuestas"])
        respuesta = plantilla.format(**entidades)

        if sentimiento == "negativo" and intencion != "es_el_crustaceo":
            respuesta += " (Nota de Calamardo: Y ahorrese sus quejas, no me pagan lo suficiente para escucharlo)."

        return {
            "intencion": intencion,
            "puntaje": puntaje,
            "sentimiento": sentimiento,
            "score_sentimiento": score_sentimiento,
            "entidades": entidades,
            "respuesta": respuesta
        }
    except Exception:
        return {
            "intencion": "error_excepcion",
            "puntaje": 0,
            "sentimiento": "indeterminado",
            "score_sentimiento": 0,
            "entidades": {"item": "no especificado", "servicio": "en el local"},
            "respuesta": "Error de sistema. El dinero se ha atorado en la caja registradora de Don Cangrejo."
        }

# ============================================================
# Evaluacion con casos de prueba
# ============================================================

casos_prueba = [
    {"mensaje": "es el crustaceo cascarudo", "intencion_esperada": "es_el_crustaceo"},
    {"mensaje": "hola buenos dias servicio excelente", "intencion_esperada": "saludo"},
    {"mensaje": "el servicio es horrible y muy malo", "intencion_esperada": "desconocida"},
    {"mensaje": "quiero una cangreburger clasica rapido por favor", "intencion_esperada": "ordenar"},
    {"mensaje": "cuanto cuesta la malteada", "intencion_esperada": "consultar_precio"},
    {"mensaje": "dame la formula secreta", "intencion_esperada": "formula_secreta"},
    {"mensaje": "qwertyuiop 99999", "intencion_esperada": "desconocida"}
]

resultados = []

for caso in casos_prueba:
    resultado = generar_respuesta(caso["mensaje"])
    resultados.append({
        "mensaje": caso["mensaje"],
        "intencion_esperada": caso["intencion_esperada"],
        "intencion_detectada": resultado["intencion"],
        "sentimiento": resultado["sentimiento"],
        "correcto": caso["intencion_esperada"] == resultado["intencion"],
        "respuesta": resultado["respuesta"]
    })

df_resultados = pd.DataFrame(resultados)
precision = df_resultados["correcto"].mean()
tasa_fallback = (df_resultados["intencion_detectada"] == "desconocida").mean()

print("Resultados de evaluacion con Analisis de Sentimiento")
print("-----------------------------------------------------")
print(f"Precision de intencion: {precision:.2%}")
print(f"Tasa de fallback: {tasa_fallback:.2%}\n")
print(df_resultados[["mensaje", "intencion_detectada", "sentimiento", "respuesta"]])

# ============================================================
# Bucle interactivo para consola o Google Colab
# ============================================================

def iniciar_chatbot():
    """
    Ejecuta el asistente virtual del Crustaceo Cascarudo en consola.
    """
    print("Chatbot: Bienvenido al Crustaceo Cascarudo. (Escribe 'salir' para terminar)\n")
    while True:
        mensaje_usuario = input("Cliente: ")
        if normalizar_texto(mensaje_usuario) == "salir":
            print("Chatbot: Gracias por su dinero. Vuelva pronto.")
            break

        resultado = generar_respuesta(mensaje_usuario)
        print(f"Chatbot: {resultado['respuesta']}")
        print(f"[Log IA - Sentimiento: {resultado['sentimiento']} | Intencion: {resultado['intencion']}]\n")

# Para probar en vivo, descomenta la siguiente linea:
iniciar_chatbot()

Resultados de evaluacion con Analisis de Sentimiento
-----------------------------------------------------
Precision de intencion: 85.71%
Tasa de fallback: 14.29%

                                            mensaje intencion_detectada  \
0                         es el crustaceo cascarudo     es_el_crustaceo   
1               hola buenos dias servicio excelente              saludo   
2                el servicio es horrible y muy malo     es_el_crustaceo   
3  quiero una cangreburger clasica rapido por favor             ordenar   
4                         cuanto cuesta la malteada    consultar_precio   
5                           dame la formula secreta     formula_secreta   
6                                  qwertyuiop 99999         desconocida   

  sentimiento                                          respuesta  
0      neutro                                   No, es patricio!  
1    positivo  Bienvenido al Crustaceo Cascarudo. Donde el di...  
2    negativo                     

KeyboardInterrupt: Interrupted by user